In [57]:
# !pip install rapidfuzz

In [11]:
from rapidfuzz import fuzz
import re
import json
from pprint import pprint
import pandas as pd
import os
# from concurrent.futures import ThreadPoolExecutor

In [12]:
# def sliding_window_score(target, candidate):
    
#     def normalize(s):
#         '''базовая нормализация'''
#         s = s.lower().replace('ё', 'е')
#         s = re.sub(r'[^\w\s\.,-:;]', ' ', s)   # всё, что не буква/цифра/пробел → пробел
#         s = re.sub(r'(?<!\d)([.,;:])(?!\d)[ \t]*', r' \1 ', s)
#         s = re.sub(r'\s+', ' ', s).strip()
#         return s 

#     target, candidate = normalize(target), normalize(candidate)

#     return {
#         "ratio": fuzz.ratio(target, candidate),                             # Классическое расстояние Левенштейна
#         "partial_ratio": fuzz.partial_ratio(target, candidate),             # Поиск лучшей подстроки (чувствителен к порядку слов)
#         "token_sort_ratio": fuzz.token_sort_ratio(target, candidate),       # Сортирует слова по алфавиту перед сравнением
#         "partial_token_sort_ratio": fuzz.partial_token_sort_ratio(target, candidate), # Сортирует слова + ищет подстроку
#         "token_set_ratio": fuzz.token_set_ratio(target, candidate),         # Игнорирует дубликаты и порядок слов (пересечение множеств)
#         "partial_token_set_ratio": fuzz.partial_token_set_ratio(target, candidate),   # То же самое, но для подстрок
#         "WRatio": fuzz.WRatio(target, candidate),                           # Взвешенное соотношение (автоматически выбирает лучший метод)
#         "QRatio": fuzz.QRatio(target, candidate),
#     }

def sliding_window_score(target, candidate, window_factor=1.5, step=1, scorer=fuzz.token_sort_ratio): #есть идеи как улучшить, смотрим на результат
    '''
    target: короткая строка
    candidate: длинная строка
    window_factor: размер окна = len(target) * window_factor
    step: шаг окна (по умолчанию = max(1, len(target)//4))
    '''
    
    def normalize(s):
        '''базовая нормализация'''
        s = s.lower().replace('ё', 'е')
        s = re.sub(r'[^\w\s\.,-:;]', ' ', s)   # всё, что не буква/цифра/пробел → пробел
        s = re.sub(r'(?<!\d)([.,;:])(?!\d)[ \t]*', r' \1 ', s)
        s = re.sub(r'\s+', ' ', s).strip()
        return s      

    target, candidate = normalize(target), normalize(candidate)

    if not window_factor:
        return scorer(target, candidate)

    best = 0.0
    
    n, m = len(target), len(candidate)
    if m <= n * window_factor or n <= 1:
        best = scorer(target, candidate)  # кандидат не длиннее окна — сравниваем целиком

    w = int(n * window_factor)


    if not best:
        for start in range(0, m - w + 1, step):
            chunk = candidate[start:start + w]
            best = max(best, scorer(target, chunk))
    return best

In [13]:
def y_to_list(obj, path = '', res=None):
    if res is None:
        res = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            y_to_list(v, path + k + '___', res)
    if isinstance(obj, list):
        for v in obj:
            y_to_list(v, path, res)
    if isinstance(obj, str):    #not isinstance(obj, (list, dict)):
        res.append([path.rstrip('_'), obj])
    if isinstance(obj, type(None)):
        res.append([path.rstrip('_'), 'None'])

    return res

# pprint(y_to_list(y))

In [14]:
def x_to_list(obj, last_path = '', res=None):
    if res is None:
        res = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            x_to_list(v, k, res)
    if isinstance(obj, list):
        for v in obj:
            x_to_list(v, last_path, res)
    if isinstance(obj, str):    #not isinstance(obj, (list, dict)):
        if last_path in ('lines', 'text'):
            res.append(obj)
    if isinstance(obj, type(None)):
        if last_path in ('lines', 'text'):
            res.append('None') 
    return res

# pprint(x_to_list(x))

In [15]:
# def create_df(x, y):
#     '''создаем df: для каждого метода fuzzy-поиска — свой лучший кандидат по X и его процент'''
#     x, y = x_to_list(x), y_to_list(y)
#     rows = []

#     methods = list(sliding_window_score('a', 'a').keys())
#     # ['ratio', 'partial_ratio', 'token_sort_ratio', 'partial_token_sort_ratio',
#     #  'token_set_ratio', 'partial_token_set_ratio', 'WRatio', 'QRatio']

#     for name, target in y:
#         row = {'обьект': name, 'по_Y': target}

#         for m in methods:
#             best_candidate = None
#             best_score = -1

#             for candidate in x:
#                 score = sliding_window_score(target, candidate)[m]
#                 if score > best_score:
#                     best_score = score
#                     best_candidate = candidate
#                 if best_score >= 99.9:
#                     break

#             row[f'по_X_{m}']   = best_candidate
#             row[f'процент_{m}'] = best_score

#         rows.append(row)

#     df = pd.DataFrame(rows)

#     # порядок колонок: обьект, по_Y, затем пара (по_X_метод, процент_метод) для каждого метода
#     cols = ['обьект', 'по_Y']
#     for m in methods:
#         cols += [f'по_X_{m}', f'процент_{m}']

#     return df[cols]

    
def create_df(x, y, all_=False):
    '''создаем df на основании листа Х и одного У'''
    
    x, y = x_to_list(x), y_to_list(y)
    
    if all_: return create_df_all(x, y)
                                  
    rows = []
    
    for name, target in y:
        best_candidate = None
        best_score = -1
        
        for candidate in x:
            score = sliding_window_score(
                target, 
                candidate, 
                window_factor=1.5, 
                step = 1,
                scorer=fuzz.token_sort_ratio
            )
            if score > best_score:
                best_score = score
                best_candidate = candidate
            
                
                if best_score >= 99.9:
                    break
                    
        rows.append([name, target, best_candidate, best_score])

    return pd.DataFrame(rows, columns=['обьект', 'по_Y', 'по_X', 'процент_нечеткого_поиска'])

def create_df_all(x, y):
    rows = []

    for name, target in y:

        for candidate in x:
            score = sliding_window_score(
                target, 
                candidate, 
                window_factor=1.5, 
                step = 1,
                scorer=fuzz.token_sort_ratio
            )

            rows.append([name, target, candidate, score])

    return pd.DataFrame(rows, columns=['обьект', 'по_Y', 'по_X', 'процент_нечеткого_поиска'])

In [16]:
from IPython.display import clear_output
# from my_worker import process_one_y

def aligment_all(path_X, file_Y, file_aligment = r'../gold/aligment_all.csv'):
    '''создание aligment.csv'''
    with open(file_Y, encoding='utf-8') as f:
        y_all = json.load(f)

    folder_path = path_X
    all_items = os.listdir(folder_path)

    df_all = []

    len_ = len(all_items)
    
    for i, item in enumerate(all_items):
        clear_output(wait=True)
        print(f'{i} из {len_} ----{i/len_*100} %')
        full_path = os.path.join(folder_path, item)
        if os.path.isfile(full_path) and item.endswith('.json'):
            act = item.split()[0].replace('.json', '')
            y = y_all[act]
            with open(full_path, encoding='utf-8') as f:
                x = json.load(f)

            df = create_df(x, y, True)
            # df['акт'] = act
            df.insert(0, 'акт', act)

            # if i == 7: break

            df_all.append(df)

    df_finish = pd.concat(df_all, ignore_index=True)

    df_finish.to_csv(file_aligment, index=False, encoding='utf-8-sig')

    return df_finish
            
df = aligment_all(r'../gold/parsed_en', r'../gold/clean.json') 

893 из 894 ----99.88814317673378 %


In [9]:
# from IPython.display import clear_output
# # from my_worker import process_one_y

# def aligment(path_X, file_Y, file_aligment = r'../gold/aligment.csv'):
#     '''создание aligment.csv'''
#     with open(file_Y, encoding='utf-8') as f:
#         y_all = json.load(f)

#     folder_path = path_X
#     all_items = os.listdir(folder_path)

#     df_all = []

#     len_ = len(all_items)
    
#     for i, item in enumerate(all_items):
#         clear_output(wait=True)
#         print(f'{i} из {len_} ----{i/len_*100} %')
#         full_path = os.path.join(folder_path, item)
#         if os.path.isfile(full_path) and item.endswith('.json'):
#             act = item.split()[0].replace('.json', '')
#             y = y_all[act]
#             with open(full_path, encoding='utf-8') as f:
#                 x = json.load(f)

#             df = create_df(x, y)
#             # df['акт'] = act
#             df.insert(0, 'акт', act)

#             # if i == 7: break

#             df_all.append(df)

#     df_finish = pd.concat(df_all, ignore_index=True)

#     df_finish.to_csv(file_aligment, index=False, encoding='utf-8-sig')

#     return df_finish
            
# df = aligment(r'../gold/parsed_en', r'../gold/clean.json') 

893 из 894 ----99.88814317673378 %


In [10]:
df

,акт,обьект,по_Y,по_X,процент_нечеткого_поиска
0,ED.A009.MII.&.012.0028,number_act,ED.A009.MII.&.012.0028,Incoming Inspection Certificate of Materials o...,80.645161
1,ED.A009.MII.&.012.0028,block,"Unit 1, 2",Unit No: 1 Date of inspection: 2022-12-07,63.636364
2,ED.A009.MII.&.012.0028,commission,Starostenko Alexander,"Type, brand, standard (technical requirement, ...",50.000000
3,ED.A009.MII.&.012.0028,commission,Obraztsova Anna,2.3. Remarks for preservation and/or anticorro...,48.648649
4,ED.A009.MII.&.012.0028,commission,Shebeshtin Alexey,2. As a result of inspection were revealed fol...,42.857143
...,...,...,...,...,...
39551,ED.A023.MII.2.012.0910-2,documents,Letter No. 003 dated 2023-02-03 (1 page),Attachments:1. Quality certificate No367 dated...,81.250000
39552,ED.A023.MII.2.012.0910-2,documents,Quality certificate No367 dated 2022-02-14 (1 ...,Attachments:1. Quality certificate No367 dated...,80.991736
39553,ED.A023.MII.2.012.0910-2,invoices,Packing list No 2730-06/2022-5 dated 2022-06-3...,Packing list No 2730-06/2022-5 dated 2022-06-30,92.156863
39554,ED.A023.MII.2.012.0910-2,projects,ED.A.A023.2.2UBN93.&&&&&&.012.KE.0002.E,ED.A.A023.2.2UBN93.&&&&&&.012.KE.0002.E; GOST ...,82.608696


# Ниже — черновик автора

In [6]:
# открыть clean.json
with open(r'../gold/clean.json', encoding='utf-8') as f:
    y_all = json.load(f)

folder_path = r'../gold/parsed_en'   #все Х

all_items = os.listdir(folder_path)

bad = []

for item in all_items:
    full_path = os.path.join(folder_path, item)
    if os.path.isfile(full_path) and item.endswith('.json'):
        try:
            act = item.split()[0].replace('.json', '')
            y = y_all[act]
        except:
            bad.append(item)
        # with open(full_path, encoding='utf-8') as f:
        #     x = json.load(f)


        
# # открыть один gold X
# with open('../gold/parsed_en/ED.A023.MII.&.&&&.02026-II (eng).json', encoding='utf-8') as f:
#     x = json.load(f)

# # y = y_all['ED.A023.MII.&.&&&.02026-II']
len(all_items)
# bad

894

In [64]:
'ED.A009.MII.&.012.0532' == "ED.A009.MII.&.012.0532"

True

In [ ]:
# def sliding_window_score(target, candidate, window_factor=1.5, step=None, scorer=fuzz.token_set_ratio): #есть идеи как улучшить, смотрим на результат
#     '''
#     target: короткая строка
#     candidate: длинная строка
#     window_factor: размер окна = len(target) * window_factor
#     step: шаг окна (по умолчанию = max(1, len(target)//4))
#     '''
    
#     def normalize(s):
#         '''базовая нормализация'''
#         s = s.lower().replace('ё', 'е')
#         s = re.sub(r'[^\w\s\.,-:;]', ' ', s)   # всё, что не буква/цифра/пробел → пробел
#         s = re.sub(r'(?<!\d)([.,;:])(?!\d)[ \t]*', r' \1 ', s)
#         s = re.sub(r'\s+', ' ', s).strip()
#         return s      

#     target, candidate = normalize(target), normalize(candidate)

#     if not window_factor:
#         return scorer(target, candidate)

#     best = 0.0
    
#     n, m = len(target), len(candidate)
#     if m <= n * window_factor:
#         best = scorer(target, candidate)  # кандидат не длиннее окна — сравниваем целиком

#     w = int(n * window_factor)
#     if step is None:
#         step = max(1, n // 4)

#     if not best:
#         for start in range(0, m - w + 1, step):
#             chunk = candidate[start:start + w]
#             best = max(best, scorer(target, chunk))
#     return best
    
# # target = 'нечёткий поиск;'
# # long_candidate = 'Это очень длинный текст, в котором где-то в середине упоминается нечёткий поиск; и потом много других слов.'

# # print(sliding_window_score(target, long_candidate, step=1))          # окно 1.5x
# # print(sliding_window_score(target, long_candidate, 2, step=1))     # окно 2x
# # print(sliding_window_score(target, long_candidate, 1.2, step=1))     # окно 1.2x — точнее, но медленнее

# print(sliding_window_score('Unit 1 . 3 . 2', 'Unit 1 . 3 . 2', step=1))
# print(sliding_window_score('x y', 'xex', step=1))

In [17]:
ord('`')

96

In [53]:
from rapidfuzz import fuzz, distance

target = 'нечёткий поиск;'
candidates = ['нечеткий понятно поиск нечеткий понятно поиск', 'поиск нечеткий;_______________________', 'поиск нечеткий; _______________________', 'нечеткийпоиск', 'поискнечеткий', 'чёткий поиск', 'поиск по тексту', 'нечёткий пойск']

for c in candidates:
    # нормализованное сходство 0..100
    # score = fuzz.ratio(target, c)
    # score = fuzz.WRatio(target, c)
    # score = fuzz.partial_token_set_ratio(target, c)
    score = sliding_window_score(target, c)
    print(f'{score:5.1f}  {c}')

 76.9  нечеткий понятно поиск нечеткий понятно поиск
 66.7  поиск нечеткий;_______________________
 66.7  поиск нечеткий; _______________________
 84.6  нечеткийпоиск
 66.7  поискнечеткий
100.0  чёткий поиск
 50.0  поиск по тексту
 92.9  нечёткий пойск


In [41]:
dir(fuzz)

['AVX2',
 'QRatio',
 'SSE2',
 'WRatio',
 '__all__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 '_impl',
 '_imported',
 'annotations',
 'contextlib',
 'os',
 'partial_ratio',
 'partial_ratio_alignment',
 'partial_token_ratio',
 'partial_token_set_ratio',
 'partial_token_sort_ratio',
 'ratio',
 'supports',
 'token_ratio',
 'token_set_ratio',
 'token_sort_ratio']

In [49]:
from rapidfuzz import fuzz

def sliding_window_score(target, candidate, window_factor=1.5, step=None, scorer=fuzz.token_sort_ratio):
    '''
    target: короткая строка
    candidate: длинная строка
    window_factor: размер окна = len(target) * window_factor
    step: шаг окна (по умолчанию = max(1, len(target)//4))
    '''
    n, m = len(target), len(candidate)
    if m <= n * window_factor:
        return scorer(target, candidate)  # кандидат не длиннее окна — сравниваем целиком

    w = int(n * window_factor)
    if step is None:
        step = max(1, n // 4)

    best = 0.0
    for start in range(0, m - w + 1, step):
        chunk = candidate[start:start + w]
        best = max(best, scorer(target, chunk))
    return best
    
target = 'нечёткий поиск;'
long_candidate = 'Это очень длинный текст, в котором где-то в середине упоминается нечёткий поиск; и потом много других слов.'

print(sliding_window_score(target, long_candidate))          # окно 1.5x
print(sliding_window_score(target, long_candidate, 2))     # окно 2x
print(sliding_window_score(target, long_candidate, 1.2))     # окно 1.2x — точнее, но медленнее

81.08108108108108
68.18181818181819
93.75


In [21]:
from rapidfuzz import fuzz, utils

# Словарь всех доступных алгоритмов в rapidfuzz.fuzz
ALL_SCORERS = {
    "ratio": fuzz.ratio,                             # Классическое расстояние Левенштейна
    "partial_ratio": fuzz.partial_ratio,             # Поиск лучшей подстроки (чувствителен к порядку слов)
    "token_sort_ratio": fuzz.token_sort_ratio,       # Сортирует слова по алфавиту перед сравнением
    "partial_token_sort_ratio": fuzz.partial_token_sort_ratio, # Сортирует слова + ищет подстроку
    "token_set_ratio": fuzz.token_set_ratio,         # Игнорирует дубликаты и порядок слов (пересечение множеств)
    "partial_token_set_ratio": fuzz.partial_token_set_ratio,   # То же самое, но для подстрок
    "WRatio": fuzz.WRatio,                           # Взвешенное соотношение (автоматически выбирает лучший метод)
    "QRatio": fuzz.QRatio,                           # Быстрое базовое соотношение (аналог ratio)
}

def test_all_fuzz_scorers(target, candidate, window_factor=1.5, step=None, preprocess=True):
    """
    Тестирует все метрики rapidfuzz с использованием скользящего окна.
    Возвращает словарь с лучшим баллом и найденным куском текста для каждой метрики.
    """
    n, m = len(target), len(candidate)
    w = int(n * window_factor)
    
    if step is None:
        step = max(1, n // 4)

    # Функция предобработки (удаляет пунктуацию типа ';' и переводит в нижний регистр)
    processor = utils.default_process if preprocess else None

    # Формируем список окон (если текст короче окна, берем текст целиком)
    if m <= w:
        windows = [(0, m, candidate)]
    else:
        windows = [
            (start, start + w, candidate[start:start + w])
            for start in range(0, m - w + 1, step)
        ]
        # Добавляем последнее окно до самого конца строки, если шаг не дошел до края
        if windows[-1][1] < m:
            windows.append((m - w, m, candidate[m - w:m]))

    results = {}

    for name, scorer in ALL_SCORERS.items():
        best_score = -1.0
        best_chunk = ""
        best_span = (0, 0)

        for start, end, chunk in windows:
            score = scorer(target, chunk, processor=processor)
            if score > best_score:
                best_score = score
                best_chunk = chunk
                best_span = (start, end)

        results[name] = {
            "score": round(best_score, 2),
            "span": best_span,
            "chunk": best_chunk
        }

    return results

# --- ТЕСТИРОВАНИЕ ---
target = 'нечёткий'
long_candidate = 'Это очень длинный текст, в котором где-то в середине упоминается нечёткий совсем поискккк и потом много других слов.'

print(f"Ищем: '{target}' (длина: {len(target)})")
print(f"В тексте длиной: {len(long_candidate)}\n")
print("-" * 80)

# Запускаем тест с предобработкой (убирает точку с запятой и регистр)
# results = test_all_fuzz_scorers(target, long_candidate, window_factor=5000, preprocess=True)
results = test_all_fuzz_scorers('5', '2134234238797068946234892529380432743298407239948', window_factor=5000, preprocess=True)

for scorer_name, data in results.items():
    print(f"{scorer_name:<25} | Балл: {data['score']:>6}% | Индексы {str(data['span']):<10} | Кусок: '{data['chunk']}'")


Ищем: 'нечёткий' (длина: 8)
В тексте длиной: 116

--------------------------------------------------------------------------------
ratio                     | Балл:    4.0% | Индексы (0, 49)    | Кусок: '2134234238797068946234892529380432743298407239948'
partial_ratio             | Балл:  100.0% | Индексы (0, 49)    | Кусок: '2134234238797068946234892529380432743298407239948'
token_sort_ratio          | Балл:    4.0% | Индексы (0, 49)    | Кусок: '2134234238797068946234892529380432743298407239948'
partial_token_sort_ratio  | Балл:  100.0% | Индексы (0, 49)    | Кусок: '2134234238797068946234892529380432743298407239948'
token_set_ratio           | Балл:    4.0% | Индексы (0, 49)    | Кусок: '2134234238797068946234892529380432743298407239948'
partial_token_set_ratio   | Балл:  100.0% | Индексы (0, 49)    | Кусок: '2134234238797068946234892529380432743298407239948'
WRatio                    | Балл:   60.0% | Индексы (0, 49)    | Кусок: '2134234238797068946234892529380432743298407239948'
Q

In [3]:
from rapidfuzz import fuzz

def sliding_window_score(target, candidate, window_factor=1.5, step=None, scorer=fuzz.partial_token_sort_ratio):
    '''
    target: короткая строка
    candidate: длинная строка
    window_factor: размер окна = len(target) * window_factor
    step: шаг окна (по умолчанию = max(1, len(target)//4))
    '''
    n, m = len(target), len(candidate)
    print('m', '---', m)
    if m <= n * window_factor:
        return scorer(target, candidate)  # кандидат не длиннее окна — сравниваем целиком

    w = int(n * window_factor)
    print('w', '---', w)
    if step is None:
        step = max(1, n // 4)
        print('step', '---', step)

    best = 0.0
    for start in range(0, m - w + 1, step):
        print('start', '---', start, '   ', 'finish', '---', start + w)
        
        chunk = candidate[start:start + w]
        best = max(best, scorer(target, chunk))
    return best
    
target = 'поиск; нечёткий'
long_candidate = 'Это очень длинный текст, в котором где-то в середине упоминается нечёткий поиск; и потом много других слов.'

print(sliding_window_score(target, long_candidate,window_factor=500, step=1))          # окно 1.5x
# print(sliding_window_score(target, long_candidate, 2))     # окно 2x
# print(sliding_window_score(target, long_candidate, 1.2))     # окно 1.2x — точнее, но медленнее

m --- 107
66.66666666666667


In [16]:
import re

def normalize(s):
    s = s.lower().replace('ё', 'е')
    s = re.sub(r'[^\w\s\.,]', ' ', s)   # всё, что не буква/цифра/пробел → пробел
    s = re.sub(r'(?<!\d)([.,])(?!\d)[ \t]*', r' \1 ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

normalize('safsed парк 123.56 sd,fdsf')

'safsed парк 123.56 sd , fdsf'